# Risk Management v2 — Trading con Alpaca (Paper Trading)

Notebook avanzado con datos reales y ejecución

## 1. Setup e instalación

Instalar librería:
`pip install alpaca-trade-api`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import alpaca_trade_api as tradeapi

## 2. Conexión a Alpaca (Paper Trading)

⚠️ Sustituye con tus credenciales

In [ ]:
API_KEY = 'YOUR_API_KEY'
API_SECRET = 'YOUR_SECRET_KEY'
BASE_URL = 'https://paper-api.alpaca.markets'

api = tradeapi.REST(API_KEY, API_SECRET, BASE_URL, api_version='v2')
account = api.get_account()
print(account.status, account.cash)

## 3. Descargar datos históricos

In [ ]:
symbol = 'AAPL'
bars = api.get_bars(symbol, tradeapi.TimeFrame.Day, limit=500).df
df = bars.copy()
df['returns'] = df['close'].pct_change()
df.head()

## 4. Cálculo ATR

In [ ]:
df['tr'] = df['high'] - df['low']
df['atr'] = df['tr'].rolling(14).mean()

## 5. Estrategia simple

In [ ]:
df['signal'] = np.where(df['returns'] > 0, 1, -1)

## 6. Risk Management

In [ ]:
capital = float(account.cash)
risk_per_trade = 0.01
entry_price = df['close'].iloc[-1]
atr = df['atr'].iloc[-1]
stop_loss = entry_price - 2*atr
risk_amount = capital * risk_per_trade
position_size = max(1, int(risk_amount / (entry_price - stop_loss)))
position_size

## 7. Backtest con costes

In [ ]:
slippage = 0.001
commission = 0.0005
df['strategy_return'] = df['returns'] * df['signal'].shift(1)
df['strategy_return'] -= slippage
df['strategy_return'] -= commission
df['equity'] = (1 + df['strategy_return']).cumprod() * capital

## 8. Drawdown

In [ ]:
df['cum_max'] = df['equity'].cummax()
df['drawdown'] = (df['cum_max'] - df['equity']) / df['cum_max']
max_dd = df['drawdown'].max()
max_dd

## 9. Kill Switch

In [ ]:
if max_dd > 0.2:
    print('STOP TRADING')
else:
    print('Riesgo OK')

## 10. Enviar orden (Paper Trading)

In [ ]:
order = api.submit_order(
    symbol=symbol,
    qty=position_size,
    side='buy',
    type='market',
    time_in_force='gtc'
)
print(order)

## 11. Visualización

In [ ]:
plt.figure()
plt.plot(df['equity'], label='Equity')
plt.legend()
plt.show()

plt.figure()
plt.plot(df['drawdown'], label='Drawdown')
plt.legend()
plt.show()